# Identifying resistive loads from single-appliance waveforms

A resistive load tends to draw current that follows the voltage waveform; motors and electronic power supplies can produce different phase and harmonic patterns. This notebook asks how well electrical measurements can distinguish **resistive from non-resistive loads** without giving a classifier appliance names, brands or locations.

**PLAID** is a public voltage/current dataset for plug-load appliance identification. Unlike household aggregate data, the recordings used here measure one appliance at a time. Each recording has waveform samples and metadata describing the appliance and recording location.

The model input is a vector of **29 electrical features** computed from voltage and current. The target is PLAID's electrical-load annotation, `load_type == "R"`; the output is a resistive-load probability followed by a fixed 0.5 decision threshold. This label describes load identity: a hair dryer annotated as resistive can be recorded with its heating element inactive.

The **`full_random_forest`** workflow covers **waveform reading → feature extraction → location-grouped training → held-out evaluation → final model fitting**. Five folds keep a recording location out of the corresponding training subset, producing out-of-fold (**OOF**) predictions for all 1,793 recordings from 55 locations. The notebook contains the extraction, model and evaluation code as well as a complete raw-data training entry.

`RUN_DATA_READ` and `RUN_TRAINING` default to `False`; default execution defines the workflow and displays recorded results. Training requires raw PLAID files and a new output directory.

[Data and dataset sources](../docs/en/data.md) · [Waveform reader](../src/load_signature_analysis/plaid.py) · [Electrical features](../src/load_signature_analysis/features.py) · [Grouped baseline](../src/load_signature_analysis/baseline.py)

The module namespace is also used by the saved model files.


In [ ]:
import sys
sys.dont_write_bytecode = True
from pathlib import Path

# Run from this checkout or its notebooks directory.
REPO = Path.cwd().resolve()
if not (REPO / "src" / "wattsign").is_dir():
    REPO = REPO.parent
if not (REPO / "src" / "wattsign").is_dir():
    raise RuntimeError("Open this notebook from the WattSign checkout.")
sys.path.insert(0, str(REPO / "src"))

from collections.abc import Mapping
from dataclasses import dataclass
import io
import json
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from wattsign.notebook_view import show_experiment


## 1. Metadata, labels and waveform access

`resistive_label = (load_type == "R")` uses PLAID's electrical-load annotation. A hair dryer marked `R` may have `status="noheat"`, so the target describes load identity rather than heater activation. `heating_active_proxy_label` is a separate label and is not used by this model.

The loader preserves sample IDs, metadata release and location groups. The CSV reader seeks backwards and reads up to the last 30,000 current/voltage pairs.


In [ ]:
@dataclass(frozen=True)
class PlaidRecord:
    sample_id: str
    release: str
    csv_path: Path
    appliance_type: str
    load_type: str
    status: str
    location: str
    brand: str
    model_number: str
    sampling_frequency_hz: float

    @property
    def resistive_label(self) -> int:
        return int(self.load_type.upper() == "R")

    @property
    def heating_active_proxy_label(self) -> int:
        # PLAID marks a hair dryer as an R-load even in its fan-only mode.
        return int(self.resistive_label == 1 and self.status.lower() != "noheat")


def _parse_frequency(value: str) -> float:
    normalized = value.strip().lower().replace("khz", "000").replace("hz", "")
    return float(normalized)


def load_plaid_records(plaid_root: Path) -> list[PlaidRecord]:
    plaid_root = Path(plaid_root)
    waveform_dir = plaid_root / "2017"
    records: list[PlaidRecord] = []
    for metadata_path in sorted(plaid_root.glob("meta_*.json")):
        release = metadata_path.stem.removeprefix("meta_")
        with metadata_path.open("r", encoding="utf-8") as handle:
            entries = json.load(handle)
        for entry in entries:
            meta = entry["meta"]
            appliance = meta["appliance"]
            sample_id = str(entry["id"])
            csv_path = waveform_dir / f"{sample_id}.csv"
            if not csv_path.exists():
                raise FileNotFoundError(f"missing PLAID waveform: {csv_path}")
            records.append(
                PlaidRecord(
                    sample_id=sample_id,
                    release=release,
                    csv_path=csv_path,
                    appliance_type=appliance.get("type", ""),
                    load_type=appliance.get("load", ""),
                    status=appliance.get("status", ""),
                    location=meta.get("location", "unknown"),
                    brand=appliance.get("brand", ""),
                    model_number=appliance.get("model_number", ""),
                    sampling_frequency_hz=_parse_frequency(meta["header"]["sampling_frequency"]),
                )
            )
    if not records:
        raise FileNotFoundError(f"no PLAID metadata found under {plaid_root}")
    return records


In [ ]:
def read_waveform_tail(csv_path: Path, max_rows: int = 30_000) -> tuple[np.ndarray, np.ndarray]:
    """Read the last rows of a two-column current/voltage CSV.

    PLAID files can be several megabytes. Seeking backwards avoids parsing the
    off-state portion when the steady-state feature extractor only needs the
    final cycles.
    """

    csv_path = Path(csv_path)
    with csv_path.open("rb") as handle:
        handle.seek(0, 2)
        file_size = handle.tell()
        block_size = min(file_size, 1_048_576)
        while True:
            handle.seek(file_size - block_size)
            payload = handle.read(block_size)
            if payload.count(b"\n") >= max_rows + 1 or block_size == file_size:
                break
            block_size = min(file_size, block_size * 2)

    lines = payload.splitlines()
    if block_size < file_size and lines:
        lines = lines[1:]
    lines = lines[-max_rows:]
    array = np.loadtxt(io.BytesIO(b"\n".join(lines)), delimiter=",", dtype=np.float64)
    if array.ndim == 1:
        array = array.reshape(1, -1)
    if array.shape[1] != 2:
        raise ValueError(f"expected two columns in {csv_path}, found {array.shape[1]}")
    current = array[:, 0]
    voltage = array[:, 1]
    return current, voltage


## 2. Electrical features

The features combine power magnitude with waveform shape. Power factor and phase describe the relationship between voltage and current; harmonic distortion and higher-order harmonics describe departures from a sinusoidal current; the V–I loop summarizes the trajectory traced by paired voltage/current samples. These complementary descriptions allow the classifier to test more than total power alone.

The 29-feature signature includes active power `mean(V * I)`, apparent power `Vrms * Irms`, absolute power factor, fundamental phase, correlation, crest/form factors, skewness/kurtosis, current total harmonic distortion (**THD**), normalized V–I loop area, linear residual and harmonics 2–15 relative to the fundamental.

Extraction estimates mains frequency from voltage FFT within 45–65 Hz, selects the final 30 cycles, removes the mean and reverses current polarity when average active power is negative. Non-finite voltage/current pairs are removed together; insufficient samples raise an error. Metadata does not enter the electrical calculation.

Feature order matches the saved pipeline. The model uses magnitude as well as waveform features, so its performance does not isolate the contribution of waveform shape.


In [ ]:
HARMONICS = (2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15)


FULL_FEATURES = (
    "mains_frequency_hz",
    "voltage_rms",
    "current_rms",
    "active_power_w",
    "apparent_power_va",
    "power_factor_abs",
    "phase_difference_deg",
    "voltage_current_correlation",
    "current_crest_factor",
    "current_form_factor",
    "current_skewness",
    "current_kurtosis",
    "current_thd",
    "vi_loop_area_normalized",
    "linear_residual_ratio",
    *tuple(f"current_h{order}_ratio" for order in HARMONICS),
)


def _rms(signal: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(signal))))


def _safe_ratio(numerator: float, denominator: float, default: float = np.nan) -> float:
    if not np.isfinite(denominator) or abs(denominator) < 1e-12:
        return default
    return float(numerator / denominator)


def estimate_mains_frequency(
    voltage: np.ndarray,
    sample_rate_hz: float,
    lower_hz: float = 45.0,
    upper_hz: float = 65.0,
) -> float:
    """Estimate the grid fundamental from the strongest voltage FFT bin."""

    centered = np.asarray(voltage, dtype=np.float64) - float(np.mean(voltage))
    if centered.size < 8:
        return np.nan
    spectrum = np.abs(np.fft.rfft(centered * np.hanning(centered.size)))
    frequencies = np.fft.rfftfreq(centered.size, d=1.0 / sample_rate_hz)
    mask = (frequencies >= lower_hz) & (frequencies <= upper_hz)
    if not np.any(mask):
        return np.nan
    candidate_indices = np.flatnonzero(mask)
    peak_index = int(candidate_indices[np.argmax(spectrum[mask])])
    return float(frequencies[peak_index])


def _complex_amplitude(signal: np.ndarray, sample_rate_hz: float, frequency_hz: float) -> complex:
    times = np.arange(signal.size, dtype=np.float64) / sample_rate_hz
    kernel = np.exp(-2j * np.pi * frequency_hz * times)
    return 2.0 * np.dot(signal, kernel) / signal.size


def _standardized_moment(signal: np.ndarray, order: int) -> float:
    centered = signal - float(np.mean(signal))
    std = float(np.std(centered))
    if std < 1e-12:
        return np.nan
    return float(np.mean(np.power(centered / std, order)))


def _normalized_vi_area(voltage: np.ndarray, current: np.ndarray, samples_per_cycle: int) -> float:
    if samples_per_cycle < 4 or voltage.size < samples_per_cycle:
        return np.nan
    v_cycle = voltage[-samples_per_cycle:].astype(np.float64, copy=True)
    i_cycle = current[-samples_per_cycle:].astype(np.float64, copy=True)
    v_scale = float(np.max(np.abs(v_cycle)))
    i_scale = float(np.max(np.abs(i_cycle)))
    if v_scale < 1e-12 or i_scale < 1e-12:
        return np.nan
    x = v_cycle / v_scale
    y = i_cycle / i_scale
    return float(0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


In [ ]:
def extract_electrical_features(
    current: np.ndarray,
    voltage: np.ndarray,
    sample_rate_hz: float,
    nominal_mains_hz: float = 60.0,
    steady_cycles: int = 30,
) -> dict[str, float]:
    """Extract power, harmonic, phase and V-I geometry features.

    The final ``steady_cycles`` are used because PLAID records an off-to-on
    transition followed by steady operation.  No appliance metadata enters the
    calculation, which keeps the feature cache suitable for grouped evaluation.
    """

    current = np.asarray(current, dtype=np.float64).reshape(-1)
    voltage = np.asarray(voltage, dtype=np.float64).reshape(-1)
    usable = min(current.size, voltage.size)
    if usable < 1_000:
        raise ValueError(f"waveform is too short: {usable} samples")
    current = current[-usable:]
    voltage = voltage[-usable:]
    finite = np.isfinite(current) & np.isfinite(voltage)
    current = current[finite]
    voltage = voltage[finite]
    if current.size < 1_000:
        raise ValueError("waveform has too few finite samples")

    frequency_hz = estimate_mains_frequency(voltage, sample_rate_hz)
    if not np.isfinite(frequency_hz):
        frequency_hz = nominal_mains_hz
    samples_per_cycle = max(4, int(round(sample_rate_hz / frequency_hz)))
    steady_count = min(current.size, steady_cycles * samples_per_cycle)
    current = current[-steady_count:] - float(np.mean(current[-steady_count:]))
    voltage = voltage[-steady_count:] - float(np.mean(voltage[-steady_count:]))

    # Orient current so that imported files with an inverted CT do not create a
    # negative active-power feature for an otherwise identical appliance.
    if float(np.mean(current * voltage)) < 0:
        current = -current

    voltage_rms = _rms(voltage)
    current_rms = _rms(current)
    active_power = float(np.mean(voltage * current))
    apparent_power = voltage_rms * current_rms
    power_factor = abs(_safe_ratio(active_power, apparent_power))

    voltage_fundamental = _complex_amplitude(voltage, sample_rate_hz, frequency_hz)
    current_fundamental = _complex_amplitude(current, sample_rate_hz, frequency_hz)
    phase_difference = np.angle(current_fundamental / voltage_fundamental, deg=True)
    phase_difference = float((phase_difference + 180.0) % 360.0 - 180.0)

    current_h1 = abs(current_fundamental)
    harmonic_amplitudes: dict[int, float] = {}
    harmonic_ratios: dict[int, float] = {}
    for order in HARMONICS:
        amplitude = abs(_complex_amplitude(current, sample_rate_hz, order * frequency_hz))
        harmonic_amplitudes[order] = float(amplitude)
        harmonic_ratios[order] = _safe_ratio(amplitude, current_h1)
    thd = _safe_ratio(
        float(np.sqrt(sum(value * value for value in harmonic_amplitudes.values()))),
        current_h1,
    )

    current_mean_abs = float(np.mean(np.abs(current)))
    correlation = float(np.corrcoef(voltage, current)[0, 1])
    linear_scale = _safe_ratio(float(np.dot(current, voltage)), float(np.dot(voltage, voltage)))
    if np.isfinite(linear_scale) and current_rms > 1e-12:
        residual_ratio = _rms(current - linear_scale * voltage) / current_rms
    else:
        residual_ratio = np.nan

    features: dict[str, float] = {
        "mains_frequency_hz": float(frequency_hz),
        "voltage_rms": voltage_rms,
        "current_rms": current_rms,
        "active_power_w": active_power,
        "apparent_power_va": apparent_power,
        "power_factor_abs": power_factor,
        "phase_difference_deg": phase_difference,
        "voltage_current_correlation": correlation,
        "current_crest_factor": _safe_ratio(float(np.max(np.abs(current))), current_rms),
        "current_form_factor": _safe_ratio(current_rms, current_mean_abs),
        "current_skewness": _standardized_moment(current, 3),
        "current_kurtosis": _standardized_moment(current, 4),
        "current_thd": thd,
        "vi_loop_area_normalized": _normalized_vi_area(voltage, current, samples_per_cycle),
        "linear_residual_ratio": float(residual_ratio),
    }
    features.update({f"current_h{order}_ratio": harmonic_ratios[order] for order in HARMONICS})
    return features


def feature_vector(features: Mapping[str, float], names: tuple[str, ...]) -> np.ndarray:
    return np.asarray([features[name] for name in names], dtype=np.float64)


## 3. Feature-frame construction

Electrical features are joined with sample metadata and labels. Only `FULL_FEATURES` enters the predictor matrix; appliance type, brand, model and location are excluded. Location defines the evaluation groups. The training workflow saves its feature cache in the configured output directory.


In [ ]:
def build_plaid_feature_cache(
    plaid_root: Path,
    output_csv: Path,
    max_rows: int = 30_000,
    limit: int | None = None,
) -> pd.DataFrame:
    records = load_plaid_records(plaid_root)
    if limit is not None:
        records = records[:limit]
    rows: list[dict[str, object]] = []
    started = time.perf_counter()
    for index, record in enumerate(records, start=1):
        current, voltage = read_waveform_tail(record.csv_path, max_rows=max_rows)
        features = extract_electrical_features(
            current=current,
            voltage=voltage,
            sample_rate_hz=record.sampling_frequency_hz,
        )
        rows.append(
            {
                "sample_id": record.sample_id,
                "release": record.release,
                "appliance_type": record.appliance_type,
                "load_type": record.load_type,
                "status": record.status,
                "location": record.location,
                "brand": record.brand,
                "model_number": record.model_number,
                "resistive_label": record.resistive_label,
                "heating_active_proxy_label": record.heating_active_proxy_label,
                **features,
            }
        )
        if index % 100 == 0 or index == len(records):
            elapsed = time.perf_counter() - started
            print(f"features: {index}/{len(records)} samples ({elapsed:.1f}s)", flush=True)
    frame = pd.DataFrame(rows)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_csv, index=False)
    return frame


## 4. Random Forest pipeline

A random forest combines the decisions of many trees, allowing it to learn nonlinear relationships between the electrical features. A median imputer supplies missing feature values using statistics fitted only on the relevant training subset.

The pipeline uses 500 trees, maximum depth 10, minimum leaf size 3, `balanced_subsample` class weighting and random seed 42. Each fold fits its imputer on the training subset. The forest receives the imputed electrical features without scaling.


In [ ]:
def make_random_forest(random_state: int = 42):
    forest = make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(
                n_estimators=500,
                max_depth=10,
                min_samples_leaf=3,
                class_weight="balanced_subsample",
                n_jobs=-1,
                random_state=random_state,
            ),
        )
    return forest


## 5. Grouped OOF evaluation

Five `GroupKFold` folds keep each location on one side of each train/test split. `clone(model)` creates an independent imputer and forest per fold. Held-out probabilities are written back to their row positions; pooled OOF metrics use a fixed 0.5 threshold. After evaluation, a final pipeline is fitted to all samples.

OOF scores come from five fold-specific models. The distributed `full_random_forest.joblib` is the final model trained on all 1,793 samples; its predictions on those same samples are in-sample predictions, not OOF evaluation.


In [ ]:
def _binary_metrics(labels: np.ndarray, probabilities: np.ndarray) -> dict[str, object]:
    predictions = (probabilities >= 0.5).astype(np.int8)
    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    return {
        "average_precision": float(average_precision_score(labels, probabilities)),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
        "confusion_matrix_tn_fp_fn_tp": [
            int(matrix[0, 0]),
            int(matrix[0, 1]),
            int(matrix[1, 0]),
            int(matrix[1, 1]),
        ],
    }


In [ ]:
def evaluate_selected_random_forest(
    frame: pd.DataFrame,
    output_dir: Path,
    target_column: str = "resistive_label",
    random_state: int = 42,
    folds: int = 5,
) -> dict[str, object]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    labels = frame[target_column].to_numpy(dtype=np.int8)
    groups = frame["location"].astype(str).to_numpy()
    splitter = GroupKFold(n_splits=folds)
    results: dict[str, object] = {
        "target": target_column,
        "samples": int(len(frame)),
        "positive_samples": int(labels.sum()),
        "positive_fraction": float(labels.mean()),
        "groups": int(pd.Series(groups).nunique()),
        "split": f"GroupKFold({folds}) by location",
        "models": {},
    }

    selected = {"full_random_forest": (FULL_FEATURES, make_random_forest(random_state))}
    for model_name, (feature_names, model) in selected.items():
        design = frame.loc[:, list(feature_names)].replace([np.inf, -np.inf], np.nan)
        probabilities = np.full(len(frame), np.nan, dtype=np.float64)
        fold_summaries: list[dict[str, object]] = []
        for fold_number, (train_indices, test_indices) in enumerate(
            splitter.split(design, labels, groups), start=1
        ):
            fitted = clone(model)
            fitted.fit(design.iloc[train_indices], labels[train_indices])
            fold_probabilities = fitted.predict_proba(design.iloc[test_indices])[:, 1]
            probabilities[test_indices] = fold_probabilities
            fold_metrics = _binary_metrics(labels[test_indices], fold_probabilities)
            fold_metrics.update(
                {
                    "fold": fold_number,
                    "train_samples": int(len(train_indices)),
                    "test_samples": int(len(test_indices)),
                    "test_groups": sorted(set(groups[test_indices].tolist())),
                }
            )
            fold_summaries.append(fold_metrics)
        metrics = _binary_metrics(labels, probabilities)
        metrics["features"] = list(feature_names)
        metrics["folds"] = fold_summaries
        results["models"][model_name] = metrics
        frame[f"oof_probability_{model_name}"] = probabilities

        final_model = clone(model)
        final_model.fit(design, labels)
        joblib.dump(
            {"model": final_model, "features": feature_names, "target": target_column},
            output_dir / f"{model_name}.joblib",
        )

    results["best_model"] = "full_random_forest"  # Use the fixed full-feature Random Forest configuration.

    prediction_columns = [
        "sample_id",
        "release",
        "location",
        "appliance_type",
        "load_type",
        "status",
        target_column,
        *[f"oof_probability_{name}" for name in results["models"]],
    ]
    frame.loc[:, prediction_columns].to_csv(output_dir / "oof_predictions.csv", index=False)
    with (output_dir / "metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(results, handle, ensure_ascii=False, indent=2)
    return results


## 6. Train from local PLAID files

Set an absolute `PLAID_ROOT`, a new absolute `OUTPUT_DIR`, `RUN_DATA_READ=True` and `RUN_TRAINING=True`. Input layout must contain `meta_*.json` and the `2017/` waveform directory. The workflow builds features, fits five fold models and a final deployment model, then saves metrics, OOF predictions and model files.

Existing output directories and paths within the dataset or a source checkout are rejected. Keep both switches `False` to inspect the implementation and recorded results without processing raw data.


In [ ]:
RUN_DATA_READ = False
RUN_TRAINING = False
PLAID_ROOT = None   # Path to the local PLAID dataset.
OUTPUT_DIR = None   # A new directory outside this checkout and input data.

if RUN_TRAINING and not RUN_DATA_READ:
    raise ValueError("Training also requires an explicit RUN_DATA_READ=True opt-in.")

if RUN_TRAINING and RUN_DATA_READ:
    if PLAID_ROOT is None or OUTPUT_DIR is None:
        raise ValueError("Explicit PLAID_ROOT and a new OUTPUT_DIR are required.")
    raw_input = Path(PLAID_ROOT).expanduser()
    raw_output = Path(OUTPUT_DIR).expanduser()
    if not raw_input.is_absolute() or not raw_output.is_absolute():
        raise ValueError("Dataset and output paths must be absolute.")
    plaid_root = raw_input.resolve(strict=True)
    output_dir = raw_output.resolve()
    if not plaid_root.is_dir():
        raise NotADirectoryError(plaid_root)
    if output_dir.exists():
        raise FileExistsError("Use a new output directory; existing results are never overwritten.")
    for protected in (plaid_root, REPO):
        if output_dir == protected or protected in output_dir.parents or output_dir in protected.parents:
            raise ValueError("Output must be separate from the dataset and public checkout.")
    for ancestor in output_dir.parents:
        if (ancestor / ".git").exists() or (
            (ancestor / "pyproject.toml").is_file() and (ancestor / "src").is_dir()
        ):
            raise ValueError("Use a dedicated artifact directory outside source checkouts.")
    if not (plaid_root / "2017").is_dir() or not any(plaid_root.glob("meta_*.json")):
        raise FileNotFoundError("Expected PLAID meta_*.json files and a 2017 waveform directory.")
    # Create the output directory before feature extraction.
    output_dir.mkdir(parents=True, exist_ok=False)
    feature_frame = build_plaid_feature_cache(plaid_root, output_dir / "features.csv")
    reproduced = evaluate_selected_random_forest(
        feature_frame, output_dir / "random_forest", target_column="resistive_label",
        random_state=42, folds=5,
    )
    print({key: reproduced[key] for key in ("samples", "groups", "split")})
else:
    print("Training: disabled (RUN_TRAINING=False).")


## 7. Results and evaluation scope

The experiment contains 1,793 samples, 55 location groups and all five folds. Pooled held-out OOF **AP 0.9553** shows that the feature/model combination separates the two load categories well in these recordings. **Recall 92.52%** is the share of resistive recordings identified at the 0.5 threshold; **F1 0.9281** summarizes precision and recall together.

This result supports using physical electrical features as a compact baseline for resistive-load classification. It does not measure when an appliance turns ON, identify each appliance type, or separate multiple appliances in a household signal.

Location grouping controls overlap between locations; it does not guarantee unseen appliance models, brands or recording environments. No confidence interval was saved for this case. Fold spread describes variation between folds and is not a confidence interval. The final model fitted to all samples is provided for reuse, while the reported evaluation comes from the separate fold models.


In [ ]:
show_experiment("plaid-resistive")


## Recorded metrics

**Average precision (AP)** summarizes the ranking of resistive versus non-resistive recordings. **Recall** is the share of resistive recordings classified correctly at the fixed threshold; **F1** combines precision and recall. These are classification metrics for held-out individual recordings.

Values below are rounded from the case summary and pool out-of-fold predictions from all five folds. The results guide defines aggregation and limitations.

### plaid-resistive

[Source summary](../results/plaid-resistive/summary.json) · [English results](../docs/en/results.md)

| Partition | Model | Backend | AP | Recall | F1 |
|---|---|---|---:|---:|---:|
| Location-grouped OOF | Random forest | — | 0.9553 | 0.9252 | 0.9281 |


## Model files, predictions and references

[Methods](../docs/en/methods.md) · [Data](../docs/en/data.md) · [Runtime and artifacts](../docs/en/runtime.md)

For access to the model and saved OOF predictions, contact **ruizhang.eric@gmail.com** with the case name, requested files and intended use. Raw PLAID recordings are obtained separately from the dataset provider.
